# Live Network IDS (FIXED VERSION)
## Proper Feature Extraction + Validation + Confidence Thresholding

In [1]:
!pip install scapy xgboost pandas numpy scikit-learn --quiet

In [1]:
import joblib
import pandas as pd
import numpy as np
import os
import sys
from datetime import datetime
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [12]:
# Load Model Files
try:
    model = joblib.load("models/model.pkl")
    scaler = joblib.load("models/scaler.pkl")
    encoders = joblib.load("models/encoders.pkl")
    target_encoder = joblib.load("models/target_encoder.pkl")
    feature_names = joblib.load("models/features.pkl")
    
    print(f"✅ Model loaded successfully")
    print(f"✅ Features: {len(feature_names)} KDD Cup features")
    print(f"✅ Target classes: {list(target_encoder.classes_)}")
    print(f"✅ Feature names: {feature_names}")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("Make sure models folder exists with: model.pkl, scaler.pkl, encoders.pkl, target_encoder.pkl, features.pkl")

c:\Users\Asad Ali\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Asad Ali\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


✅ Model loaded successfully
✅ Features: 41 KDD Cup features
✅ Target classes: ['attack', 'normal']
✅ Feature names: ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate']


In [3]:
# Import custom modules
from feature_aggregator import NetworkFeatureAggregator
from validation_layer import PredictionValidator, PredictionPostProcessor

print("✅ Custom modules imported")

✅ Custom modules imported


In [4]:
def encode_categorical_features(features_df):
    """Encode categorical KDD Cup features before validation and scaling."""
    encoded_df = features_df.copy()

    categorical_columns = ['protocol_type', 'service', 'flag']
    for column in categorical_columns:
        if column in encoded_df.columns:
            value = encoded_df[column].iloc[0]
            encoder = encoders.get(column)

            if encoder is not None and value in encoder.classes_:
                encoded_df[column] = encoder.transform([value])[0]
            else:
                encoded_df[column] = 0

    return encoded_df


In [ ]:
# Initialize components
aggregator = NetworkFeatureAggregator(
    window_size=10,  # 10 second window for aggregation
    feature_names=feature_names
)

validator = PredictionValidator(
    feature_names=feature_names
)

post_processor = PredictionPostProcessor(
    confidence_threshold=0.85,  # Only predict if confidence > 85%
    attack_probability_threshold=0.70,  # Flag as attack if prob > 70%
    attack_class_index=0  # Training label encoder mapped 'attack' to class 0
)

print("✅ Components initialized")
print(f"   - Feature Aggregator (window: 10s)")
print(f"   - Validation Layer")
print(f"   - Post Processor (confidence: 85%, attack_prob: 70%)")

✅ Components initialized
   - Feature Aggregator (window: 10s)
   - Validation Layer
   - Post Processor (confidence: 85%, attack_prob: 70%)


In [6]:
# Statistics
stats = {
    'packets_processed': 0,
    'predictions_made': 0,
    'attacks_detected': 0,
    'normal_traffic': 0,
    'uncertain': 0,
    'validation_failed': 0,
    'start_time': datetime.now()
}

print("✅ Statistics tracking initialized")

✅ Statistics tracking initialized


In [7]:
from scapy.all import sniff, IP

def process_packet(packet):
    """
    Process a single network packet through the complete pipeline:
    1. Feature extraction with aggregation
    2. Feature validation
    3. Model prediction
    4. Post-processing with thresholding
    """
    stats['packets_processed'] += 1
    
    try:
        # Step 1: Skip non-IP packets
        if not packet.haslayer(IP):
            return
        
        # Step 2: Extract features with aggregation
        features_df = aggregator.extract_features(packet)
        if features_df is None:
            return

        # Step 2.5: Encode categorical columns for validation/scaling/model input
        features_df = encode_categorical_features(features_df)
        
        # Step 3: Validate features
        is_valid, validation_report = validator.validate(features_df)
        
        if not is_valid:
            stats['validation_failed'] += 1
            print(f"\n❌ Packet #{stats['packets_processed']}: VALIDATION FAILED")
            validator.log_report(validation_report)
            return
        
        # Step 4: Prepare data for model
        # Ensure columns are in correct order
        features_df = features_df[feature_names]
        
        # Scale features
        features_scaled = scaler.transform(features_df)
        
        # Step 5: Get model prediction
        prediction = model.predict(features_scaled)[0]
        probabilities = model.predict_proba(features_scaled)[0]
        
        # Step 6: Post-process prediction
        result = post_processor.process_prediction(
            prediction=prediction,
            probabilities=probabilities,
            validation_report=validation_report
        )
        
        # Step 7: Update statistics and print result
        stats['predictions_made'] += 1
        
        if result['final_prediction'] == 'ATTACK':
            stats['attacks_detected'] += 1
        elif result['final_prediction'] == 'NORMAL':
            stats['normal_traffic'] += 1
        elif result['final_prediction'] == 'UNCERTAIN':
            stats['uncertain'] += 1
        
        # Print result
        post_processor.print_prediction(result)
        
        # Every 20 packets, print statistics
        if stats['packets_processed'] % 20 == 0:
            print_statistics()
    
    except Exception as e:
        print(f"❌ Error processing packet: {e}")
        import traceback
        traceback.print_exc()

print("✅ process_packet function defined")

✅ process_packet function defined


In [8]:
def print_statistics():
    """Print session statistics"""
    elapsed = (datetime.now() - stats['start_time']).total_seconds()
    
    print("\n" + "="*70)
    print(f"📊 STATISTICS (elapsed: {elapsed:.1f}s)")
    print("="*70)
    print(f"  Total packets processed:     {stats['packets_processed']}")
    print(f"  Predictions made:            {stats['predictions_made']}")
    print(f"  🚨 Attacks detected:          {stats['attacks_detected']}")
    print(f"  ✅ Normal traffic:            {stats['normal_traffic']}")
    print(f"  ⚠️  Uncertain:                 {stats['uncertain']}")
    print(f"  ❌ Validation failed:         {stats['validation_failed']}")
    
    if stats['predictions_made'] > 0:
        attack_rate = (stats['attacks_detected'] / stats['predictions_made']) * 100
        normal_rate = (stats['normal_traffic'] / stats['predictions_made']) * 100
        uncertain_rate = (stats['uncertain'] / stats['predictions_made']) * 100
        print(f"\n  Attack rate:  {attack_rate:.1f}%")
        print(f"  Normal rate:  {normal_rate:.1f}%")
        print(f"  Uncertain:    {uncertain_rate:.1f}%")
    
    print("="*70 + "\n")

print("✅ print_statistics function defined")

✅ print_statistics function defined


## 🚀 Start Live Traffic Analysis

**IMPORTANT:**
- Run this cell to start sniffing network traffic
- May require root/admin privileges
- Press Ctrl+C to stop
- Statistics will be printed every 20 packets

In [13]:
print("🔴 STARTING LIVE NETWORK ANALYSIS...")
print("Press Ctrl+C to stop\n")

try:
    sniff(
        prn=process_packet,
        store=False,
        filter="ip"
    )
except KeyboardInterrupt:
    print("\n🔴 Stopped by user")
    print_statistics()
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\nNote: Packet sniffing requires root/admin privileges")

🔴 STARTING LIVE NETWORK ANALYSIS...
Press Ctrl+C to stop



2026-05-25 15:51:09,542 - WARNING - ⚠️ SUSPICIOUS: 26 out of 41 features are zero! This might be a data quality issue.
2026-05-25 15:51:09,545 - INFO - ✅ Features validation passed



⚠️  [WARNING] ATTACK
   Normal: 1.29% | Attack: 98.71%
   Confidence: 98.71%
   • Attack detected with 98.71% confidence
   • ⚠️ 1 validation warnings


2026-05-25 15:51:11,311 - WARNING - ⚠️ SUSPICIOUS: 27 out of 41 features are zero! This might be a data quality issue.
2026-05-25 15:51:11,314 - INFO - ✅ Features validation passed



⚠️  [WARNING] ATTACK
   Normal: 0.01% | Attack: 99.99%
   Confidence: 99.99%
   • Attack detected with 99.99% confidence
   • ⚠️ 1 validation warnings


...


⚠️  [WARNING] ATTACK
   Normal: 2.28% | Attack: 97.72%
   Confidence: 97.72%
   • Attack detected with 97.72% confidence
   • ⚠️ 1 validation warnings


2026-05-25 15:51:58,782 - WARNING - ⚠️ SUSPICIOUS: 26 out of 41 features are zero! This might be a data quality issue.
2026-05-25 15:51:58,784 - INFO - ✅ Features validation passed



⚠️  [WARNING] ATTACK
   Normal: 2.28% | Attack: 97.72%
   Confidence: 97.72%
   • Attack detected with 97.72% confidence
   • ⚠️ 1 validation warnings


2026-05-25 15:51:58,930 - WARNING - ⚠️ SUSPICIOUS: 26 out of 41 features are zero! This might be a data quality issue.
2026-05-25 15:51:58,933 - INFO - ✅ Features validation passed



⚠️  [WARNING] ATTACK
   Normal: 1.73% | Attack: 98.27%
   Confidence: 98.27%
   • Attack detected with 98.27% confidence
   • ⚠️ 1 validation warnings


## 📈 Testing with Sample Packets

If you can't run live sniffing, test with example network traffic

In [9]:
# Example: Create test packets
# This is for testing without actual network traffic

from scapy.all import IP, TCP, UDP

print("🧪 Creating test packets...\n")

# Test 1: Normal HTTP request
test_packet_1 = IP(dst="8.8.8.8")/TCP(dport=80, flags="S")
print(f"Test 1: Normal HTTP packet")
process_packet(test_packet_1)

# Test 2: Another normal request
test_packet_2 = IP(dst="8.8.8.8")/TCP(dport=443, flags="S")
print(f"\nTest 2: HTTPS packet")
process_packet(test_packet_2)

# Test 3: DNS query
test_packet_3 = IP(dst="8.8.8.8")/UDP(dport=53)
print(f"\nTest 3: DNS packet")
process_packet(test_packet_3)

print("\n✅ Test packets processed")
print_statistics()

🧪 Creating test packets...

Test 1: Normal HTTP packet


2026-05-25 15:45:09,845 - WARNING - ⚠️ SUSPICIOUS: 24 out of 41 features are zero! This might be a data quality issue.
2026-05-25 15:45:09,850 - INFO - ✅ Features validation passed



⚠️  [WARNING] UNCERTAIN
   Normal: 69.32% | Attack: 30.68%
   Confidence: 69.32%
   • Low confidence (69.32%). Requires manual review.
   • ⚠️ 1 validation warnings

Test 2: HTTPS packet


2026-05-25 15:45:10,142 - WARNING - ⚠️ SUSPICIOUS: 25 out of 41 features are zero! This might be a data quality issue.
2026-05-25 15:45:10,144 - INFO - ✅ Features validation passed



⚠️  [WARNING] UNCERTAIN
   Normal: 39.79% | Attack: 60.21%
   Confidence: 60.21%
   • Low confidence (60.21%). Requires manual review.
   • ⚠️ 1 validation warnings

Test 3: DNS packet


2026-05-25 15:45:10,240 - WARNING - ⚠️ SUSPICIOUS: 26 out of 41 features are zero! This might be a data quality issue.
2026-05-25 15:45:10,243 - INFO - ✅ Features validation passed



⚠️  [WARNING] ATTACK
   Normal: 8.68% | Attack: 91.32%
   Confidence: 91.32%
   • Attack detected with 91.32% confidence
   • ⚠️ 1 validation warnings

✅ Test packets processed

📊 STATISTICS (elapsed: 26.6s)
  Total packets processed:     3
  Predictions made:            3
  🚨 Attacks detected:          1
  ✅ Normal traffic:            0
  ⚠️  Uncertain:                 2
  ❌ Validation failed:         0

  Attack rate:  33.3%
  Normal rate:  0.0%
  Uncertain:    66.7%



## 📊 Final Statistics

In [ ]:
print_statistics()

# Print recent predictions
if post_processor.prediction_log:
    print("\n📋 RECENT PREDICTIONS (last 10):\n")
    for i, pred in enumerate(post_processor.prediction_log[-10:], 1):
        print(f"{i}. [{pred['severity']}] {pred['final_prediction']} "
              f"(Normal: {pred['normal_probability']:.1%}, "
              f"Attack: {pred['attack_probability']:.1%}, "
              f"Confidence: {pred['confidence']:.1%})")


📊 STATISTICS (elapsed: 189.7s)
  Total packets processed:     201
  Predictions made:            201
  🚨 Attacks detected:          136
  ✅ Normal traffic:            13
  ⚠️  Uncertain:                 52
  ❌ Validation failed:         0

  Attack rate:  67.7%
  Normal rate:  6.5%
  Uncertain:    25.9%


📋 RECENT PREDICTIONS (last 10):

1. [WARNING] ATTACK (Normal: 0.8%, Attack: 99.2%, Confidence: 99.2%)
2. [WARNING] ATTACK (Normal: 0.8%, Attack: 99.2%, Confidence: 99.2%)
3. [WARNING] ATTACK (Normal: 0.8%, Attack: 99.2%, Confidence: 99.2%)
4. [WARNING] ATTACK (Normal: 0.8%, Attack: 99.2%, Confidence: 99.2%)
5. [WARNING] ATTACK (Normal: 0.2%, Attack: 99.8%, Confidence: 99.8%)
6. [WARNING] UNCERTAIN (Normal: 49.9%, Attack: 50.1%, Confidence: 50.1%)
7. [WARNING] ATTACK (Normal: 0.8%, Attack: 99.2%, Confidence: 99.2%)
8. [WARNING] ATTACK (Normal: 0.8%, Attack: 99.2%, Confidence: 99.2%)
9. [WARNING] ATTACK (Normal: 0.8%, Attack: 99.2%, Confidence: 99.2%)
10. [WARNING] ATTACK (Normal: 0.8%